# 📅 Scheduling Snowflake dbt Pipelines with Prefect

---

## 🤔 Why Schedule dbt on Snowflake?

| Schedule Pattern | Use Case |
|---|---|
| Daily at 6 AM | Refresh Snowflake tables every morning for BI reports |
| Every 4 hours | Near-realtime analytics on Snowflake |
| First of month | Monthly aggregates / billing reports |
| Weekdays at 8 AM | Business hours Snowflake refresh |

---

## 💻 Example 1: The Flow We Will Schedule

In [ ]:
from prefect import flow, task
from prefect.blocks.system import Secret
from datetime import date
import subprocess, os

DBT_PROJECT_DIR = "/Users/aviraljain/Downloads/python advanced/my_etl_project"

def get_snowflake_env():
    env = os.environ.copy()
    for env_key, block_name in {
        "SNOWFLAKE_ACCOUNT":  "snowflake-account",
        "SNOWFLAKE_USER":     "snowflake-user",
        "SNOWFLAKE_PASSWORD": "snowflake-password",
    }.items():
        try: env[env_key] = Secret.load(block_name).get()
        except Exception: pass
    return env

def run_dbt(command: str, target: str = "dev", extra: list = None):
    cmd = ["dbt"] + command.split() + ["--target", target,
           "--project-dir", DBT_PROJECT_DIR, "--profiles-dir", DBT_PROJECT_DIR]
    if extra: cmd += extra
    result = subprocess.run(cmd, capture_output=True, text=True, env=get_snowflake_env())
    print(result.stdout[-500:])
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-300:])

@task(name="dbt seed ❄️", retries=1)
def dbt_seed(target): run_dbt("seed", target)

@task(name="dbt run ❄️", retries=2, retry_delay_seconds=30)
def dbt_run(target, full_refresh=False):
    run_dbt("run", target, ["--full-refresh"] if full_refresh else None)

@task(name="dbt test ❄️")
def dbt_test(target): run_dbt("test", target)


@flow(name="Snowflake dbt — Scheduled", log_prints=True)
def snowflake_dbt_pipeline(
    run_date: str = str(date.today()),
    target: str = "dev",
    full_refresh: bool = False
):
    """The flow to be scheduled. All params can be overridden at runtime."""
    print(f"📅 Run date: {run_date} | ❄️ Target: {target} | 🔄 Full refresh: {full_refresh}")
    dbt_seed(target)
    dbt_run(target, full_refresh)
    dbt_test(target)
    print(f"✅ Snowflake pipeline complete for {run_date}!")

# Manual test run
snowflake_dbt_pipeline(target="dev")

---

## 💻 Example 2: `.serve()` — Quick Local Scheduling

In [ ]:
# ⚠️ Uncomment to start → this BLOCKS the notebook (keeps running until Ctrl+C)

# snowflake_dbt_pipeline.serve(
#     name="snowflake-dbt-daily",
#     cron="0 6 * * *",           # Every day at 6 AM
#     timezone="Asia/Kolkata",    # IST
#     parameters={                 # Default params passed at each scheduled run
#         "target": "prod",
#         "full_refresh": False
#     }
# )

print("Common Snowflake scheduling patterns:")
schedules = {
    "0 6 * * *":    "Daily at 6 AM — morning Snowflake refresh before office",
    "0 */4 * * *":  "Every 4 hours — near-realtime analytics",
    "0 8 * * 1-5": "Weekdays 8 AM — business hours only (saves Snowflake credits)",
    "0 0 1 * *":   "Monthly — billing aggregates or monthly reports",
}
for cron, desc in schedules.items():
    print(f"  {cron:<18} → {desc}")

---

## 💻 Example 3: `.deploy()` — Production with Prefect Worker

In [ ]:
# Production deployment (run as a .py file, not in notebook)
deploy_code = '''
if __name__ == "__main__":
    snowflake_dbt_pipeline.deploy(
        name="prod-snowflake-dbt-daily",
        work_pool_name="local-pool",
        cron="0 6 * * *",
        timezone="Asia/Kolkata",
        parameters={
            "target": "prod",
            "full_refresh": False
        },
        tags=["dbt", "snowflake", "production"],
        description="Daily Snowflake dbt refresh — seed, run, test"
    )
'''
print("Production deployment script:")
print(deploy_code)

print("\nTerminal setup:")
steps = [
    "prefect work-pool create local-pool --type process",
    "prefect worker start --pool local-pool",
    "python snowflake_dbt_pipeline.py   # Creates deployment in Prefect Cloud"
]
for i, s in enumerate(steps, 1):
    print(f"  Step {i}: {s}")

---

## 💻 Example 4: Snowflake Credit Optimization via Smart Scheduling

In [ ]:
# Strategy: Only run full_refresh on Sundays (to save Snowflake credits weekdays)
from datetime import date

@flow(name="Smart Snowflake dbt", log_prints=True)
def smart_snowflake_pipeline(run_date: str = str(date.today())):
    """Automatically decides if today needs a full-refresh based on day of week."""
    today = date.fromisoformat(run_date)
    is_sunday = today.weekday() == 6   # 6 = Sunday

    print(f"📅 {run_date} | Day: {today.strftime('%A')} | Full refresh: {is_sunday}")

    dbt_seed(target="dev")
    dbt_run(target="dev", full_refresh=is_sunday)  # Full refresh only on Sundays
    dbt_test(target="dev")

    print(f"✅ Smart Snowflake pipeline done! Full-refresh: {is_sunday}")

smart_snowflake_pipeline()

---

## 🏭 Summary

| Method | Best For | Snowflake Consideration |
|---|---|---|
| Manual call | Testing/debugging | Use `target='dev'` |
| `.serve(cron=)` | Local schedule | Machine must stay on |
| `.deploy(cron=)` | Production | Use `target='prod'`, secrets |
| Day-based full-refresh | Credit optimization | `--full-refresh` only when needed |

---

## ⚠️ Common Beginners' Mistakes

In [ ]:
mistakes = [
    ("full_refresh=True every run",       "Rebuilds all Snowflake tables every time — very expensive!"),
    ("No timezone in schedule",           "Cron runs in UTC by default — your 6 AM becomes 12:30 AM IST"),
    ("target='dev' in production",        "Always use target='prod' for scheduled production runs"),
    ("Scheduling too frequently",         "Every 15 min on Snowflake = lots of warehouse credits consumed"),
]
for mistake, fix in mistakes:
    print(f"❌ {mistake}")
    print(f"✅ Fix: {fix}\n")